# Core types and money

**Purpose:** Learn finstack's typed primitives for currencies, cash amounts, rates, identifiers, metadata, and library configuration.

*No prerequisites — this is the first notebook in the curriculum.*


## Why financial type safety matters

In production finance code, raw floats and strings are easy to misuse: adding dollars to euros, confusing percent with decimal, or treating identifiers as interchangeable. finstack wraps these concepts in small, explicit types so invalid combinations fail early and APIs stay self-documenting.


## `Currency` — ISO-4217 currency codes

Immutable currency values with alphabetic and numeric ISO codes, decimals for formatting, and JSON helpers.


In [1]:
from finstack.core.currency import Currency, USD, EUR, GBP, JPY

c = Currency("USD")
c_num = Currency.from_numeric(840)
print("from string:", c.code, c.numeric, c.decimals)
print("from numeric:", c_num.code, c_num.numeric)

j = c.to_json()
print("to_json:", j)
print("from_json:", Currency.from_json(j).code)

print("constants:", USD.code, EUR.code, GBP.code, JPY.code)
print("compare Currency:", USD == EUR, USD != EUR)
print("ordering (alphabetic ISO codes):", EUR < USD, GBP > EUR)
print("compare with str:", Currency("USD") == "USD")


from string: USD 840 2
from numeric: USD 840
to_json: "USD"
from_json: USD
constants: USD EUR GBP JPY
compare Currency: False True
ordering (alphabetic ISO codes): False False
compare with str: True


## `Money` — amounts tagged with a currency

Construct with a string or `Currency`, format for display, serialize, and do arithmetic — mixing currencies raises `ValueError`.


In [2]:
from finstack.core.currency import Currency
from finstack.core.money import Money

m1 = Money(100.0, "USD")
m2 = Money(100.0, Currency("USD"))
z = Money.zero("USD")
print("constructors equal:", m1 == m2)
print("zero:", z.amount, z.currency.code)
print("amount / currency:", m1.amount, m1.currency)
print("format default:", m1.format())
print("format 4dp:", m1.format(decimals=4))

js = m1.to_json()
print("json round-trip:", Money.from_json(js))
t = m1.to_tuple()
print("tuple:", t, "from_tuple:", Money.from_tuple(t))

x = m1 + Money(50.0, "USD")
y = x - Money(25.0, "USD")
print("add/sub:", x, y)
print("mul/div/neg:", m1 * 1.5, m1 / 2.0, -m1)

try:
    _ = Money(1.0, "USD") + Money(1.0, "EUR")
except ValueError as e:
    print("cross-currency add:", type(e).__name__, str(e))


constructors equal: True
zero: 0.0 USD
amount / currency: 100.0 USD
format default: USD 100.00
format 4dp: USD 100.0000
json round-trip: USD 100.00
tuple: (100.0, 'USD') from_tuple: USD 100.00
add/sub: USD 150.00 USD 125.00
mul/div/neg: USD 150.00 USD 50.00 USD -100.00
cross-currency add: ValueError Currency mismatch: expected USD, got EUR


## `Rate` — decimal, percent, and basis points

A single rate type with conversions and safe arithmetic.


In [3]:
from finstack.core.types import Rate

r = Rate(0.05)
r_pct = Rate.from_percent(5.0)
r_bps = Rate.from_bps(500)
print("as_decimal:", r.as_decimal)
print("as_percent:", r.as_percent)
print("as_bps:", r.as_bps)
print("from_percent matches:", r == r_pct)
print("from_bps matches:", r == r_bps)
print("ZERO:", Rate.ZERO.as_decimal)

a = Rate(0.03) + Rate(0.02)
b = Rate(0.10) - Rate(0.02)
c = Rate(0.04) * 2.0
d = Rate(0.10) / 2.0
print("arithmetic:", a.as_decimal, b.as_decimal, c.as_decimal, d.as_decimal)


as_decimal: 0.05
as_percent: 5.0
as_bps: 500
from_percent matches: True
from_bps matches: True
ZERO: 0.0
arithmetic: 0.05 0.08 0.08 0.05


## `Bps` — basis points (1 bp = 0.01%)

Integer-rounded basis-point amounts with decimal conversion.


In [4]:
from finstack.core.types import Bps

b = Bps(250)
print("as_decimal:", b.as_decimal)
print("as_bps:", b.as_bps)
print("ZERO:", Bps.ZERO.as_bps)
print("arithmetic:", (Bps(100) + Bps(50)).as_bps, (Bps(300) - Bps(50)).as_bps)


as_decimal: 0.025
as_bps: 250
ZERO: 0
arithmetic: 150 250


## `Percentage` — human percent values

Stores values like `12.5` meaning 12.5%.


In [5]:
from finstack.core.types import Percentage

p = Percentage(12.5)
print("as_decimal:", p.as_decimal)
print("as_percent:", p.as_percent)
print("ZERO:", Percentage.ZERO.as_percent)


as_decimal: 0.125
as_percent: 12.5
ZERO: 0.0


## `CreditRating` — normalized rating categories

Use class constants or parse S&P/Fitch-style and Moody's strings; notches map to the base bucket.


In [6]:
from finstack.core.types import CreditRating

print("AAA:", CreditRating.AAA.name)
r1 = CreditRating.from_name("BBB")
r2 = CreditRating.from_name("BBB+")
r3 = CreditRating.from_name("Baa1")
print("from_name BBB:", r1.name)
print("from_name BBB+:", r2.name)
print("from_name Baa1:", r3.name)


AAA: AAA
from_name BBB: BBB
from_name BBB+: BBB+
from_name Baa1: BBB+


## `CurveId` and `InstrumentId` — stable string identifiers

Lightweight wrappers for curve and instrument keys used across market data and instruments.


In [7]:
from finstack.core.types import CurveId, InstrumentId

cid = CurveId("USD-OIS")
iid = InstrumentId("BOND-001")
print("CurveId:", cid.as_str)
print("InstrumentId:", iid.as_str)


CurveId: USD-OIS
InstrumentId: BOND-001


## `Attributes` — string metadata bag

Attach key/value tags (all strings) to instruments or structures.


In [8]:
from finstack.core.types import Attributes

attrs = Attributes()
attrs.set("sector", "Technology")
attrs.set("rating", "BBB")
print("get sector:", attrs.get("sector"))
print("contains sector:", attrs.contains("sector"))
print("keys:", attrs.keys())
print("len:", attrs.len())


get sector: Technology
contains sector: True
keys: ['rating', 'sector']
len: 2


## `RoundingMode`, `ToleranceConfig`, and `FinstackConfig`

Control rounding semantics, numerical tolerances, and per-currency output scales.


In [9]:
from finstack.core.config import RoundingMode, ToleranceConfig, FinstackConfig

rm = RoundingMode.BANKERS
rm2 = RoundingMode.from_name("bankers")
print("rounding modes equal:", rm == rm2)

tc0 = ToleranceConfig()
tc1 = ToleranceConfig(rate_epsilon=1e-10)
print("default rate_epsilon:", tc0.rate_epsilon)
print("custom rate_epsilon:", tc1.rate_epsilon)

cfg0 = FinstackConfig()
cfg1 = FinstackConfig(rounding_mode=RoundingMode.BANKERS)
print("default output_scale USD/JPY:", cfg0.output_scale("USD"), cfg0.output_scale("JPY"))
print("with bankers rounding repr:", cfg1)


rounding modes equal: True
default rate_epsilon: 1e-12
custom rate_epsilon: 1e-10
default output_scale USD/JPY: 2 0
with bankers rounding repr: FinstackConfig(...)


## Mini-example: multi-currency cash position tracker

Track balances in three currencies, apply deposits and fees, print formatted balances, and show cross-currency mistakes raising errors.


In [10]:
from finstack.core.money import Money

positions = {
    "USD": Money.zero("USD"),
    "EUR": Money.zero("EUR"),
    "JPY": Money.zero("JPY"),
}

def deposit(ccy: str, amt: float) -> None:
    positions[ccy] = positions[ccy] + Money(amt, ccy)

def fee(ccy: str, amt: float) -> None:
    positions[ccy] = positions[ccy] - Money(amt, ccy)

deposit("USD", 1_000_000.0)
deposit("EUR", 250_000.0)
deposit("JPY", 50_000_000.0)
fee("USD", 1_250.50)
fee("EUR", 300.0)

print("--- Balances ---")
for ccy in ("USD", "EUR", "JPY"):
    print(positions[ccy].format())

usd_net = positions["USD"] + Money(10_000.0, "USD")
print("USD after inflow:", usd_net.format())

print("--- Cross-currency guard ---")
try:
    _ = positions["USD"] + positions["EUR"]
except ValueError as e:
    print("blocked:", e)


--- Balances ---
USD 998749.50
EUR 249700.00
JPY 50000000
USD after inflow: USD 1008749.50
--- Cross-currency guard ---
blocked: Currency mismatch: expected USD, got EUR


## Takeaways

- **`Currency` / `Money`** encode ISO currencies and tagged amounts; arithmetic refuses currency mismatches.
- **`Rate`, `Bps`, `Percentage`** give consistent conversions between decimal, percent, and basis-point views.
- **`CreditRating`, `CurveId`, `InstrumentId`, `Attributes`** standardize ratings, IDs, and metadata strings.
- **`FinstackConfig`** ties rounding and tolerances to library-wide behavior, including output precision by currency.

**Next:** Continue the foundations track with dates and calendars, or jump ahead to instruments once these primitives feel familiar.
